# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）** を使い、`person` の1クラスの検出モデルを作成します。

このノートブックでは、まずGPU・Google Drive・保存済みファイルだけを確認します。
各セルを上から順に実行してください。**セル4の出力を確認してから、次の処理を追加します。**
画像展開、ZIP内容の検証、注釈解析、ラベル変換、モデル取得、学習はまだ行いません。

- データ保存場所: `/content/drive/MyDrive/crowdhuman`
- 学習結果の保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重みの保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- 今後のローカルデータ展開先: `/content/crowdhuman_vbox_yolo`（現段階では作成しません）


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. 保存先とColabの空き容量を確認

学習結果用と事前学習重み用のフォルダを作成します。既存のファイルは削除しません。
データ保存フォルダとローカルデータ展開先は、このセルでは作成しません。
表示する空き容量は **Colabローカル** の容量です。


In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "crowdhuman"
OUTPUT_ROOT = DRIVE_ROOT / "crowdhuman_vbox_ft"
RUNS_DIR = OUTPUT_ROOT / "runs"
WEIGHTS_DIR = OUTPUT_ROOT / "weights"
LOCAL_DATA_DIR = Path("/content/crowdhuman_vbox_yolo")

if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Driveが接続されていません。先にセル2を実行してください。")

for directory in (RUNS_DIR, WEIGHTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"データ保存場所: {DATA_DIR}")
print(f"学習結果の保存先: {RUNS_DIR}")
print(f"事前学習重みの保存先: {WEIGHTS_DIR}")
print(f"今後のローカルデータ展開先: {LOCAL_DATA_DIR}")
usage = shutil.disk_usage("/content")
print(f"Colabローカルの総容量: {usage.total / 1024**3:.2f} GiB")
print(f"Colabローカルの空き容量: {usage.free / 1024**3:.2f} GiB")


## 4. CrowdHumanの保存済みファイルを確認

データフォルダ直下を最大50件表示し、想定する配布ファイルの存在と空ファイルでないことを確認します。
このセルは読み取りのみで、ZIPの展開・内容の検証や注釈の読み込みは行いません。

ファイルが見つからなくても、展開済みや別名保存の可能性があります。
**実行結果を共有し、確認できるまでここで止めてください。**


In [ ]:
EXPECTED_FILES = (
    "CrowdHuman_train01.zip",
    "CrowdHuman_train02.zip",
    "CrowdHuman_train03.zip",
    "CrowdHuman_val.zip",
    "annotation_train.odgt",
    "annotation_val.odgt",
)

if not DATA_DIR.is_dir():
    print(f"データフォルダが見つかりません: {DATA_DIR}")
    print("Driveの接続状態と保存場所を確認してください。")
else:
    entries = sorted(DATA_DIR.iterdir(), key=lambda path: path.name)
    print(f"データフォルダ直下の一覧（全{len(entries)}件、最大50件表示）:")
    for entry in entries[:50]:
        if entry.is_dir():
            print(f"  [フォルダ] {entry.name}")
        elif entry.is_file():
            print(f"  [ファイル] {entry.name} ({entry.stat().st_size:,} bytes)")
        else:
            print(f"  [その他] {entry.name}")
    if len(entries) > 50:
        print(f"  残り{len(entries) - 50}件は省略しました。")

    print("\n想定する配布ファイルの確認:")
    for name in EXPECTED_FILES:
        path = DATA_DIR / name
        if not path.exists():
            print(f"  [未検出] {name}")
        elif not path.is_file():
            print(f"  [通常ファイルではありません] {name}")
        else:
            size = path.stat().st_size
            status = "空ファイル" if size == 0 else "存在・サイズ確認済み"
            print(f"  [{status}] {name}: {size:,} bytes")

    print("\nこの確認ではファイル内容の完全性は検証していません。")
    print("未検出の場合は、展開済み・別名保存の可能性も含めて出力を確認します。")

print("\nここで実行結果を共有してください。次の処理はまだ追加していません。")
